# OpsShield — Quantum Incident Response Optimizer

**Hackathon track:** Open-Domain Optimization & Real-World Solvers

This notebook demonstrates OpsShield's infrastructure remediation
optimization model, exact classical baseline, QUBO formulation, and
Qiskit QAOA experiment.

The dataset is synthetic. QAOA is evaluated using local simulation;
this notebook does not claim quantum hardware execution or quantum advantage.


In [ ]:
from pathlib import Path
import os
import sys
import pprint

# Locate the OpsShield project root, whether the notebook opens
# from the repository root or the notebooks directory.
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (
    (ROOT / "pyproject.toml").exists()
    and (ROOT / "src" / "opshield").exists()
):
    ROOT = ROOT.parent

if not ((ROOT / "src" / "opshield").exists()):
    raise RuntimeError(
        "Could not locate the OpsShield project root. "
        "Open this notebook from the OpsShield workspace."
    )

os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import qiskit
from opshield.model import (
    ACTIONS,
    exact_classical_solution,
    qubo_coefficients,
    qubo_to_hamiltonian,
)
from opshield.solver import run_qaoa

print("Project root:", ROOT)
print("Qiskit version:", qiskit.__version__)
print("Notebook working directory:", Path.cwd())


## 1. Remediation planning problem

OpsShield selects infrastructure remediation actions subject to an
engineering effort budget and incompatibility constraints.

The objective is to maximize total remediation benefit while respecting
the effort budget and avoiding incompatible action pairs.


In [ ]:
print("Available remediation actions:")
pprint.pp(ACTIONS)


## 2. Exact classical baseline

An exhaustive search over the 64 possible portfolios of six binary
actions provides a classical reference solution.


In [ ]:
baseline = exact_classical_solution()

print("Selected action IDs:", baseline["selected_ids"])
print("Selected actions:", baseline["selected_names"])
print("Total effort:", baseline["effort"], "/ 7")
print("Total benefit:", baseline["benefit"])

assert baseline["effort"] <= 7
assert baseline["benefit"] == 28


## 3. QUBO formulation

The binary optimization problem is represented by a Quadratic
Unconstrained Binary Optimization (QUBO) objective. A penalty coefficient
discourages budget violations and incompatible action selections.

The following cell inspects the QUBO coefficients and corresponding
Hamiltonian generated by the project's implementation.


In [ ]:
PENALTY = 144

constant, linear, quadratic = qubo_coefficients(penalty=PENALTY)

print("Penalty coefficient:", PENALTY)
print("QUBO constant:", constant)
print("Number of linear coefficients:", len(linear))
print("Linear coefficients:", linear.tolist())
print("Quadratic coefficients:")
pprint.pp(quadratic)

hamiltonian = qubo_to_hamiltonian(penalty=PENALTY)
print("\nHamiltonian type:", type(hamiltonian).__name__)
if hasattr(hamiltonian, "num_qubits"):
    print("Hamiltonian qubits:", hamiltonian.num_qubits)
print(hamiltonian)


## 4. Execute the QAOA experiment

Run the existing OpsShield solver using Qiskit's QAOA implementation
and local simulation. The parameters below use depth 1, a maximum of
60 optimizer iterations, 4,096 shots and random seed 7.

Generated reports and charts are written to `artifacts/notebook_demo/`.


In [ ]:
from IPython.display import display

ARTIFACT_DIR = ROOT / "artifacts" / "notebook_demo"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

qaoa_result = run_qaoa(
    reps=1,
    maxiter=60,
    shots=4096,
    seed=7,
    output_dir=str(ARTIFACT_DIR),
    penalty=PENALTY,
)

print("QAOA result:")
pprint.pp(qaoa_result)


## 5. Inspect generated reports and plots

The next cell displays generated JSON reports and PNG charts, if present,
so the saved notebook can retain visible evidence of the experiment.


In [ ]:
import json
from IPython.display import Image, Markdown, display

generated_files = sorted(
    p for p in ARTIFACT_DIR.rglob("*") if p.is_file()
)

print("Artifact directory:", ARTIFACT_DIR)
print("Generated file count:", len(generated_files))

for file in generated_files:
    print("\nFile:", file.relative_to(ROOT))

    if file.suffix.lower() == ".png":
        display(Image(filename=str(file)))
    elif file.suffix.lower() == ".json":
        try:
            with file.open(encoding="utf-8") as stream:
                report = json.load(stream)
            print(json.dumps(report, indent=2, default=str))
        except (ValueError, OSError) as exc:
            print("Could not display JSON report:", exc)
    elif file.suffix.lower() == ".csv":
        print(file.read_text(encoding="utf-8")[:12000])

if not generated_files:
    print("No files were generated in the artifact directory.")


## 6. Interpretation and limitations

- The exact classical baseline provides a reference for evaluating QAOA.
- QAOA performance depends on parameters, sampling and random seed.
- Synthetic inputs do not establish real-world operational performance.
- Local simulation does not demonstrate execution on quantum hardware.
- No quantum advantage is claimed.

For reproducibility, this notebook records the parameters used for the
QAOA experiment and retains cell outputs after execution.
